# Testing the SING coefficient-count formula

This notebook checks the dense SING parameterisation used in `sing_rvine_10D.ipynb` against

$$P_{\mathrm{SING}}=P_c+P_h=\binom{D+B+1}{B+1}-1.$$

Here $D\geq1$ is dimensionality and $B\geq1$ is the map order (`ORDER` in the R-vine notebook). For a dense triangular map, component $k$ has a degree-$B$ offset in $k-1$ variables and a degree-$(B-1)$ integrand in $k$ variables, giving

$$P_c=\sum_{k=1}^{D}\binom{k-1+B}{B}=\binom{D+B}{B+1},
\qquad
P_h=\sum_{k=1}^{D}\binom{k+B-1}{B-1}=\binom{D+B}{B}-1.$$

Pascal's identity yields the stated total. The numerical tests independently compare these formulas with the coefficients allocated by TransportMaps and the dimensions of its actual parameter Hessian. They provide implementation checks for a finite set of configurations; the combinatorial identities explain the general formula.

Select the **Python (GTM SING)** kernel and run all cells. No data generation, fitting, or graph recovery is needed. This formula concerns the **initial dense map**; sparse later iterations can have fewer coefficients.

In [1]:
from itertools import product
from math import comb
from time import perf_counter
import importlib.metadata as metadata

import numpy as np
import pandas as pd
from IPython.display import display
from TransportMaps.Maps import assemble_IsotropicIntegratedSquaredTriangularTransportMap
from TransportMaps.Distributions import StandardNormalDistribution
from TransportMaps.TransportMapDistributions import PullBackParametricTransportMapDistribution

print('TransportMaps:', metadata.version('TransportMaps'))
print('NumPy:', np.__version__)

TransportMaps: 3.2.0
NumPy: 2.2.6


## Configurations

The default grid combines $D\in\{1,2,3,5,10\}$ with $B\in\{1,2,3\}$. Additional cases cover $B=10$ at smaller dimensions, higher-order maps, and $D=20$. This includes the original $D=10,B=2$ example.

Edit the lists below to add cases. The Hessian is evaluated at one point, so its output alone requires $8P^2$ bytes in double precision. The configurable memory limit applies to this output tensor; intermediate calculations can require additional memory.

In [2]:
DIMENSIONS = [1, 2, 3, 5, 10]
DEGREES = [1, 2, 3]
EXTRA_CASES = [(1, 10), (2, 10), (3, 10), (5, 5), (10, 4), (20, 1), (20, 2)]
MAX_HESSIAN_MIB = 128

CASES = sorted(set(product(DIMENSIONS, DEGREES)).union(EXTRA_CASES))
print(f'{len(CASES)} distinct (D, B) combinations: {CASES}')

22 distinct (D, B) combinations: [(1, 1), (1, 2), (1, 3), (1, 10), (2, 1), (2, 2), (2, 3), (2, 10), (3, 1), (3, 2), (3, 3), (3, 10), (5, 1), (5, 2), (5, 3), (5, 5), (10, 1), (10, 2), (10, 3), (10, 4), (20, 1), (20, 2)]


## Test method

For each case, construct the same dense integrated-squared triangular map as the R-vine diagnostic (`active_vars=None`, `btype='fun'`) and initialise it to the identity. Count $c$ and $h$ coefficients directly from the map components, then call `hess_a_log_pdf` at one point.

The returned tensor must have shape $(1,P,P)$. Infer the observed total from its last dimension and compare it with the map's total coefficient count, the sum of its $c$ and $h$ counts, and the binomial formula. Check the two component formulas separately as well. All entries of the probe Hessian must be finite. The tensor is discarded after extracting its dimensions.

In [3]:
def check_coefficient_count(D, B, max_hessian_mib=MAX_HESSIAN_MIB):
    for name, value in {'D': D, 'B': B}.items():
        if isinstance(value, bool) or not isinstance(value, int) or value < 1:
            raise ValueError(f'{name} must be a positive integer')

    expected_c = comb(D + B, B + 1)
    expected_h = comb(D + B, B) - 1
    expected_total = comb(D + B + 1, B + 1) - 1
    predicted_bytes = expected_total ** 2 * np.dtype(np.float64).itemsize
    if predicted_bytes > max_hessian_mib * 2 ** 20:
        raise ValueError(f'D={D}, B={B}: Hessian output would require '
                         f'{predicted_bytes / 2**20:.1f} MiB, above the '
                         f'{max_hessian_mib} MiB limit.')

    model = assemble_IsotropicIntegratedSquaredTriangularTransportMap(
        D, B, active_vars=None, btype='fun')
    model.coeffs = model.get_identity_coeffs()
    actual_c = int(sum(component.c.n_coeffs for component in model.approx_list))
    actual_h = int(sum(component.h.n_coeffs for component in model.approx_list))
    actual_map_total = int(model.n_coeffs)
    density = PullBackParametricTransportMapDistribution(model, StandardNormalDistribution(D))
    hessian = density.hess_a_log_pdf(np.zeros((1, D)))
    shape = tuple(int(size) for size in hessian.shape)
    if len(shape) != 3 or shape[0] != 1 or shape[1] != shape[2]:
        raise AssertionError(f'D={D}, B={B}: unexpected Hessian shape {shape}')
    finite = bool(np.isfinite(hessian).all())
    observed_total = shape[-1]
    del hessian

    c_matches = actual_c == expected_c
    h_matches = actual_h == expected_h
    total_matches = observed_total == actual_map_total == actual_c + actual_h == expected_total
    return dict(D=D, B=B, P_c=actual_c, P_c_formula=expected_c,
                P_h=actual_h, P_h_formula=expected_h, P_from_hessian=observed_total,
                P_from_map=actual_map_total, P_formula=expected_total, Hessian_shape=shape,
                c_matches=c_matches, h_matches=h_matches, total_matches=total_matches,
                finite_hessian=finite, passed=bool(c_matches and h_matches and total_matches and finite))

In [4]:
started = perf_counter()
results = pd.DataFrame([check_coefficient_count(D, B) for D, B in CASES])
elapsed_seconds = perf_counter() - started

summary = results.set_index(['D', 'B'])[
    ['P_c', 'P_h', 'Hessian_shape', 'P_from_hessian', 'P_formula', 'passed']]
display(summary)
failures = results.loc[~results['passed']]
if not failures.empty:
    display(failures)
assert failures.empty, 'At least one coefficient-count check failed; inspect failures above.'
print(f"PASS: all {len(results)} configurations match P_c, P_h, and P_SING.")
print(f'Elapsed: {elapsed_seconds:.2f} seconds')

PASS: all 22 configurations match P_c, P_h, and P_SING.
Elapsed: 177.90 seconds


P_c   P_h    Hessian_shape  P_from_hessian  P_formula  passed
D  B                                                                 
1  1      1     1        (1, 2, 2)               2          2    True
   2      1     2        (1, 3, 3)               3          3    True
   3      1     3        (1, 4, 4)               4          4    True
   10     1    10      (1, 11, 11)              11         11    True
2  1      3     2        (1, 5, 5)               5          5    True
   2      4     5        (1, 9, 9)               9          9    True
   3      5     9      (1, 14, 14)              14         14    True
   10    12    65      (1, 77, 77)              77         77    True
3  1      6     3        (1, 9, 9)               9          9    True
   2     10     9      (1, 19, 19)              19         19    True
   3     15    19      (1, 34, 34)              34         34    True
   10    78   285    (1, 363, 363)             363        363    True
5  1     15     5      (1, 20, 20)              20         20    True
   2     35    20      (1, 55, 55)              55         55    True
   3     70    55    (1, 125, 125)             125        125    True
   5    210   251    (1, 461, 461)             461        461    True
10 1     55    10      (1, 65, 65)              65         65    True
   2    220    65    (1, 285, 285)             285        285    True
   3    715   285  (1, 1000, 1000)            1000       1000    True
   4   2002  1000  (1, 3002, 3002)            3002       3002    True
20 1    210    20    (1, 230, 230)             230        230    True
   2   1540   230  (1, 1770, 1770)            1770       1770    True

## The 10-dimensional reference case

For $D=10$ and $B=2$, the expected counts are $P_c=220$, $P_h=65$, and $P_{\mathrm{SING}}=285$. The parameter-Hessian tensor should therefore have dimensions $(1,285,285)$. The following cell reports the observed values when this case is included in the test list.

The complete `results` DataFrame also retains all formula counts and individual checks for inspection.

In [5]:
reference = results.loc[(results['D'] == 10) & (results['B'] == 2)]
if reference.empty:
    print('Add (10, 2) to CASES to include the reference example.')
else:
    display(reference.set_index(['D', 'B'])[
        ['P_c', 'P_c_formula', 'P_h', 'P_h_formula', 'Hessian_shape', 'P_formula', 'passed']])

,,P_c,P_c_formula,P_h,P_h_formula,Hessian_shape,P_formula,passed
D,B,,,,,,,
10,2,220,220,65,65,"(1, 285, 285)",285,True
